# Demonstrating secure communication using ML-KEM
This notebook demonstrates how Alice and Bob can use ML-KEM to securely negotiate a shared secret key over an insecure channel, and then use that key to send a message.
**Hardware Acceleration Edition**: We benchmark the pure software stack vs the custom Zynq FPGA hardware accelerator, offloading all NTT/INTT calculations via AXI DMA.

In [1]:
import sys
import os
import time
import hashlib
sys.path.append(os.path.abspath('/home/xilinx/jupyter_notebooks/EE712/Project/ml-kem'))


from mlkem.ml_kem import ML_KEM
from mlkem.parameter_set import ML_KEM_768

# Import MODULE OBJECTS so we can patch them at the source
import mlkem.auxiliary.ntt as ntt_module
import mlkem.k_pke as k_pke_module

from mlkem.auxiliary.ntt import ntt as sw_ntt, ntt_inv as sw_intt
from mlkem.auxiliary.ntt_hardware import ntt as hw_ntt, ntt_inv as hw_intt
import mlkem.auxiliary.ntt_hardware as hw_module

print("All modules loaded.")

Loading ML-KEM NTT Hardware Overlay...


Hardware loaded successfully!
All modules loaded.


In [2]:
def patch_ntt(ntt_fn, ntt_inv_fn):
    """Correctly patches both k_pke's namespace and the source ntt module."""
    k_pke_module.ntt = ntt_fn
    k_pke_module.ntt_inv = ntt_inv_fn
    ntt_module.ntt = ntt_fn
    ntt_module.ntt_inv = ntt_inv_fn

print("Patch helper defined.")

Patch helper defined.


### Step 1: Alice generates a keypair, Bob encapsulates, Alice decapsulates
First, we run the entire protocol using the pure Python software reference implementation to establish a baseline.

In [3]:
print("--- Starting Pure Software Protocol ---")
patch_ntt(sw_ntt, sw_intt)

ml_kem = ML_KEM(ML_KEM_768, fast=False)

start_sw = time.time()
ek_sw, dk_sw = ml_kem.key_gen()
k_bob_sw, c_sw = ml_kem.encaps(ek_sw)
k_alice_sw = ml_kem.decaps(dk_sw, c_sw)
end_sw = time.time()
time_sw = (end_sw - start_sw) * 1000

assert k_alice_sw == k_bob_sw, "Software keys do not match!"
print(f"Software protocol complete in: {time_sw:.2f} ms")
print(f"SW Shared Secret: {k_alice_sw.hex()}")

--- Starting Pure Software Protocol ---
Software protocol complete in: 3114.25 ms
SW Shared Secret: 20255106d4196e2afbc975d2a67f4cbef8bbc225f3e16c885a5353ca1606e744


### Step 2: Protocol with FPGA Hardware Acceleration
Now we inject the custom PYNQ overlay driver. The `ntt_hardware.py` module uses an AXI DMA to rapidly stream polynomial coefficients to the logic fabric, offloading the most computationally expensive part of the protocol.

In [4]:
print("\n--- Starting Hardware Accelerated Protocol ---")
hw_module.HW_CALL_LOG.clear()  # Reset diagnostic log
patch_ntt(hw_ntt, hw_intt)

start_hw = time.time()
ek_hw, dk_hw = ml_kem.key_gen()

k_bob_hw, c_hw = ml_kem.encaps(ek_hw)
k_alice_hw = ml_kem.decaps(dk_hw, c_hw)
end_hw = time.time()
time_hw = (end_hw - start_hw) * 1000

assert k_alice_hw == k_bob_hw, "Hardware keys do not match!"
print(f"Hardware protocol complete in: {time_hw:.2f} ms")
print(f"HW Shared Secret: {k_alice_hw.hex()}")

speedup = time_sw / time_hw if time_hw > 0 else 0
print(f"\n>>> Protocol Speedup with Hardware NTT: {speedup:.2f}x <<<")


--- Starting Hardware Accelerated Protocol ---
Hardware protocol complete in: 2545.03 ms
HW Shared Secret: 975027291765637ba7bef6913654d160239dd061f7e2188429582b230c96f2ec

>>> Protocol Speedup with Hardware NTT: 1.22x <<<


### Step 3: Hybrid Interoperability Test
To prove mathematical perfection, we will verify that a key generated in hardware can be decrypted by software, and vice-versa.

In [5]:
log = hw_module.HW_CALL_LOG
n_calls = len(log)

print(f"\n=== Hardware NTT/INTT Call Breakdown ===")
print(f"Total calls made during protocol: {n_calls}")
print(f"{'#':<4} {'fn':<6} {'t_fill':>8} {'t_send':>8} {'t_recv':>8} {'t_obj':>8} {'total':>8}  (ms)")
print("-" * 58)
for i, entry in enumerate(log):
    print(f"{i+1:<4} {entry['fn']:<6} {entry['t_fill']:>8.2f} {entry['t_send']:>8.2f} {entry['t_recv']:>8.2f} {entry['t_obj']:>8.2f} {entry['total']:>8.2f}")

totals = {k: sum(e[k] for e in log) for k in ['t_fill','t_send','t_recv','t_obj','total']}
print("-" * 58)
print(f"{'SUM':<10} {totals['t_fill']:>8.2f} {totals['t_send']:>8.2f} {totals['t_recv']:>8.2f} {totals['t_obj']:>8.2f} {totals['total']:>8.2f}")
print(f"\nHardware NTT total time consumed: {totals['total']:.2f} ms out of {time_hw:.2f} ms protocol")
print(f"Everything else (SHA3, encode/decode, etc): {time_hw - totals['total']:.2f} ms")


=== Hardware NTT/INTT Call Breakdown ===
Total calls made during protocol: 24
#    fn       t_fill   t_send   t_recv    t_obj    total  (ms)
----------------------------------------------------------
1    ntt        0.71     0.65     0.68     1.77     3.80
2    ntt        0.61     0.51     0.50     2.83     4.45
3    ntt        0.64     0.55     0.50     2.12     3.81
4    ntt        0.63     0.56     0.51     1.78     3.48
5    ntt        0.99     0.54     0.51     2.01     4.04
6    ntt        0.62     0.53     0.50     1.87     3.51
7    ntt        0.75     0.63     0.52     2.58     4.49
8    ntt        0.70     0.73     0.53     6.24     8.20
9    ntt        0.70     0.58     0.51     1.76     3.56
10   intt       0.76     0.62     0.54     1.66     3.59
11   intt       0.69     0.59     0.50     2.05     3.83
12   intt       0.67     0.52     0.50     3.17     4.86
13   intt       0.75     0.61     0.51     1.62     3.50
14   ntt        0.75     0.61     0.54     1.69     3.59
1

### Step 4: Bob and Alice begin secure communication
Now that Alice and Bob share `k_alicae = k_bob`, they can use a symmetric cipher to encrypt communication between the two of them. We use the hardware-accelerated shared secret as the seed for SHAKE-256.

In [6]:
print("\n--- Verifying Correctness Across Hardware & Software ---")

patch_ntt(hw_ntt, hw_intt)
k_bob_hybrid, c_hybrid = ml_kem.encaps(ek_sw)

patch_ntt(sw_ntt, sw_intt)
k_alice_hybrid = ml_kem.decaps(dk_sw, c_hybrid)

if k_bob_hybrid == k_alice_hybrid:
    print("✅ SUCCESS: Hybrid HW-SW protocol generated identical secrets!")
    print("The custom FPGA accelerator strictly conforms to FIPS-203.")
else:
    print("❌ FAILURE: Hardware and software did not interoperate correctly.")


--- Verifying Correctness Across Hardware & Software ---
✅ SUCCESS: Hybrid HW-SW protocol generated identical secrets!
The custom FPGA accelerator strictly conforms to FIPS-203.


In [7]:
def stream_cipher_crypt(key: bytes, message: bytes) -> bytes:
    key_stream = hashlib.shake_256(key).digest(len(message))
    return bytes([m ^ k for m, k in zip(message, key_stream)])

bobs_message = b"Hello Alice! ML-KEM successfully secured this channel with Zynq FPGA Acceleration!"
print(f"Bob's original message: {bobs_message.decode()}")

encrypted_message = stream_cipher_crypt(k_bob_hw, bobs_message)
print(f"Encrypted bits sent over channel: {encrypted_message.hex()[:40]}...")

decrypted_message = stream_cipher_crypt(k_alice_hw, encrypted_message)
print(f"Alice decrypted: {decrypted_message.decode()}")
assert decrypted_message == bobs_message
print("Message successfully sent and received!")

Bob's original message: Hello Alice! ML-KEM successfully secured this channel with Zynq FPGA Acceleration!
Encrypted bits sent over channel: e534f1df3fcc6ff01233d90c0e3098dfdbb08f71...
Alice decrypted: Hello Alice! ML-KEM successfully secured this channel with Zynq FPGA Acceleration!
Message successfully sent and received!
